# Sysslan IT Solutions - Train Schedule Analysis Project

## Level 1: Basic Data Review

In [1]:
pip install pandas matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
mkdir charts

A subdirectory or file charts already exists.


In [3]:
import pandas as pd

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [5]:
data_path = "Dataset1.csv"

In [6]:
def load_dataset(path):
    return pd.read_csv(path)

In [7]:
df = load_dataset(data_path)

### Task 1.1: Overview of dataset - total records & attributes.

In [8]:
def task_1_1_overview(df):
    print("TASK 1.1 - DATASET OVERVIEW")
    print(f"Total records: {df.shape[0]:,}")
    print(f"Total attributes: {df.shape[1]}")
    print(df.dtypes)
    print(df.head())

### Task 1.2: Start and end station for every train.

In [9]:
def task_1_2_train_routes(df):
    ordered = df.sort_values(["Train_No", "SN"])
    starts = ordered.groupby("Train_No").first()["Station_Name"]
    ends = ordered.groupby("Train_No").last()["Station_Name"]
    route_summary = pd.DataFrame({
        "Start_Station": starts,
        "End_Station": ends
    }).reset_index()
    print("\nTASK 1.2 - START & END STATIONS")
    print(route_summary.head(10))
    return route_summary

### Task 1.3: Number of stops per train.

In [10]:
def task_1_3_stops_per_train(df):
    stops = df.groupby("Train_No").size().reset_index(name="Total_Stops")
    print("\nTASK 1.3 - STOPS PER TRAIN")
    print(stops.describe())
    return stops

### Task 1.4: Trains with the most and fewest stops.

In [11]:
def task_1_4_max_min_stops(stops):
    max_row = stops.loc[stops["Total_Stops"].idxmax()]
    min_row = stops.loc[stops["Total_Stops"].idxmin()]
    print("\nTASK 1.4 - MAX / MIN STOPS")
    print(f"Max: Train {max_row['Train_No']} -> {max_row['Total_Stops']} stops")
    print(f"Min: Train {min_row['Train_No']} -> {min_row['Total_Stops']} stops")
if __name__ == "__main__":
    df = load_dataset(data_path)
    task_1_1_overview(df)
    task_1_2_train_routes(df)
    stops = task_1_3_stops_per_train(df)
    task_1_4_max_min_stops(stops)

TASK 1.1 - DATASET OVERVIEW
Total records: 186,074
Total attributes: 12
SN                 int64
Train_No           int64
Station_Code      object
1A                 int64
2A                 int64
3A                 int64
SL                 int64
Station_Name      object
Route_Number       int64
Arrival_time      object
Departure_Time    object
Distance           int64
dtype: object
   SN  Train_No Station_Code   1A   2A   3A   SL  Station_Name  Route_Number Arrival_time Departure_Time  Distance
0   1       107          SWV  100  100  100  100  SAWANTWADI R             1     00:00:00       10:25:00         0
1   2       107         THVM  260  228  196  164        THIVIM             1     11:06:00       11:08:00        32
2   3       107         KRMI  345  296  247  198       KARMALI             1     11:28:00       11:30:00        49
3   4       107          MAO  490  412  334  256   MADGOAN JN.             1     12:10:00       00:00:00        78
4   1       108          MAO  100  100 

## Level 2 — Simple Data Processing

### Task 2.1: Convert time strings to timedelta.

In [12]:
def task_2_1_standardize_times(df):
    df = df.copy()
    df["Arrival_td"] = pd.to_timedelta(df["Arrival_time"])
    df["Departure_td"] = pd.to_timedelta(df["Departure_Time"])

    is_first = df["SN"] == df.groupby("Train_No")["SN"].transform("min")
    is_last = df["SN"] == df.groupby("Train_No")["SN"].transform("max")
    df.loc[is_first, "Arrival_td"] = pd.NaT
    df.loc[is_last, "Departure_td"] = pd.NaT
    return df

### Task 2.2: Total journey duration per train, allowing for overnight rollovers (when the clock appears to go backwards).

In [13]:
def task_2_2_journey_duration(df):
    ordered = df.sort_values(["Train_No", "SN"]).copy()

    def compute_duration(group):
        start_time = group.iloc[0]["Departure_td"]
        end_time = group.iloc[-1]["Arrival_td"]
        if pd.isna(start_time) or pd.isna(end_time):
            return pd.NaT
        total = pd.Timedelta(0)
        prev_clock = start_time
        clocks = group["Arrival_td"].fillna(group["Departure_td"]).tolist()[1:]
        clocks.append(end_time)
        for clock in clocks:
            if pd.isna(clock):
                continue
            if clock < prev_clock:
                total += pd.Timedelta(days=1)
            prev_clock = clock
        return total + (end_time - start_time)

    durations = ordered.groupby("Train_No").apply(compute_duration, include_groups=False)
    duration_df = durations.reset_index()
    duration_df.columns = ["Train_No", "Journey_Duration"]
    duration_df["Journey_Duration_Hours"] = (
        duration_df["Journey_Duration"].dt.total_seconds() / 3600
    ).round(2)
    print("TASK 2.2 - JOURNEY DURATION")
    print(duration_df.head(10))
    return duration_df

### Task 2.3: Short <= 4h, Medium 4-12h, Long > 12h.

In [14]:
def task_2_3_classify_routes(duration_df):
    def classify(hours):
        if pd.isna(hours):
            return "Unknown"
        if hours <= 4:
            return "Short"
        elif hours <= 12:
            return "Medium"
        return "Long"

    duration_df = duration_df.copy()
    duration_df["Route_Type"] = duration_df["Journey_Duration_Hours"].apply(classify)
    print("\nTASK 2.3 - ROUTE TYPE COUNTS")
    print(duration_df["Route_Type"].value_counts())
    return duration_df

### Task 2.4: How many distinct trains serve each station.

In [15]:
def task_2_4_station_frequency(df):
    freq = (
        df.groupby("Station_Name")["Train_No"].nunique()
        .reset_index(name="Train_Count")
        .sort_values("Train_Count", ascending=False)
    )
    print("\nTASK 2.4 - TOP 10 BUSIEST STATIONS")
    print(freq.head(10))
    return freq


if __name__ == "__main__":
    df = load_dataset(data_path)
    df = task_2_1_standardize_times(df)
    duration_df = task_2_2_journey_duration(df)
    duration_df = task_2_3_classify_routes(duration_df)
    task_2_4_station_frequency(df)
    duration_df.to_csv("train_durations_and_route_types.csv", index=False)
    print("\nSaved train_durations_and_route_types.csv")

TASK 2.2 - JOURNEY DURATION
   Train_No Journey_Duration  Journey_Duration_Hours
0       107  0 days 01:45:00                    1.75
1       108  0 days 01:55:00                    1.92
2       128  0 days 22:05:00                   22.08
3       290  5 days 08:00:00                  128.00
4       401  1 days 12:30:00                   36.50
5       421  1 days 09:00:00                   33.00
6       422  1 days 02:45:00                   26.75
7       477  2 days 03:10:00                   51.17
8       502  0 days 23:00:00                   23.00
9       504  1 days 01:00:00                   25.00

TASK 2.3 - ROUTE TYPE COUNTS
Route_Type
Short     7107
Long      2050
Medium    1956
Name: count, dtype: int64

TASK 2.4 - TOP 10 BUSIEST STATIONS
      Station_Name  Train_Count
1755    CST-MUMBAI         1027
3466     KALYAN JN          828
7509         THANE          796
6834       SEALDAH          745
1592  CHENNAI BEAC          738
2950    HOWRAH JN.          699
1773         DADA

## Level 3 — Data Quality Checks

In [16]:
clean_path = "train_schedule_cleaned.csv"

### Task 3.1: Report and drop rows with missing key fields.

In [17]:
def task_3_1_missing_values(df):
    print("TASK 3.1 - MISSING VALUES")
    print(df.isnull().sum())
    key_cols = ["Train_No", "SN", "Station_Code", "Station_Name",
                "Arrival_time", "Departure_Time"]
    before = len(df)
    df = df.dropna(subset=key_cols)
    print(f"Rows dropped for missing key fields: {before - len(df)}")

    blank = (
        df["Station_Code"].astype(str).str.strip().eq("")
        | df["Station_Name"].astype(str).str.strip().eq("")
    )
    df = df[~blank]
    return df

### Task 3.2: Drop fully identical rows and duplicate (Train_No, Station_Code, SN) combinations.

In [18]:
def task_3_2_remove_duplicates(df):
    print("\nTASK 3.2 - DUPLICATES")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    df = df.drop_duplicates()
    dupes = df.duplicated(subset=["Train_No", "Station_Code", "SN"])
    print(f"Duplicate (Train_No, Station_Code, SN): {dupes.sum()}")
    df = df[~dupes]
    print(f"Row count after cleanup: {len(df):,}")
    return df

### Task 3.3: Flag trains whose SN sequence isn't clean(1, 2, 3, ... with no gaps or repeats).

In [19]:
def task_3_3_verify_station_order(df):
    def is_ordered(sn_series):
        sn_sorted = sn_series.sort_values().tolist()
        return sn_sorted == list(range(1, len(sn_sorted) + 1))

    check = df.groupby("Train_No")["SN"].apply(is_ordered)
    bad = check[~check].index.tolist()
    print("\nTASK 3.3 - STATION ORDER CHECK")
    print(f"Total trains checked: {len(check):,}")
    print(f"Clean trains: {check.sum():,} / {len(check):,}")
    print(f"Flagged trains: {len(bad)} -> {bad[:10]}")
    return df

### Task 3.4: Save the verified dataset.

In [20]:
def task_3_4_save_cleaned(df):
    df = df.sort_values(["Train_No", "SN"]).reset_index(drop=True)
    df.to_csv(clean_path, index=False)
    print(f"\nTASK 3.4 - Saved cleaned dataset to {clean_path}")
    print(f"Final shape: {df.shape}")

if __name__ == "__main__":
    df = load_dataset(data_path)
    df = task_3_1_missing_values(df)
    df = task_3_2_remove_duplicates(df)
    df = task_3_3_verify_station_order(df)
    task_3_4_save_cleaned(df)

TASK 3.1 - MISSING VALUES
SN                0
Train_No          0
Station_Code      0
1A                0
2A                0
3A                0
SL                0
Station_Name      0
Route_Number      0
Arrival_time      0
Departure_Time    0
Distance          0
dtype: int64
Rows dropped for missing key fields: 0

TASK 3.2 - DUPLICATES
Exact duplicate rows: 0
Duplicate (Train_No, Station_Code, SN): 0
Row count after cleanup: 186,074

TASK 3.3 - STATION ORDER CHECK
Total trains checked: 11,113
Clean trains: 11,107 / 11,113
Flagged trains: 6 -> [6919, 6920, 12978, 56905, 56906, 57641]

TASK 3.4 - Saved cleaned dataset to train_schedule_cleaned.csv
Final shape: (186074, 12)


## Level 4 — Basic Analysis and Visualization

In [21]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
sns.set_theme(style="whitegrid")

In [23]:
cleaned_path = "train_schedule_cleaned.csv"
durations_path = "train_durations_and_route_types.csv"
chart_dir = "charts"

### Task 4.1: Average journey duration per route type.

In [24]:
def task_4_1_avg_duration_by_route_type(durations):
    avg = (
        durations.groupby("Route_Type")["Journey_Duration_Hours"]
        .mean().round(2).reindex(["Short", "Medium", "Long"])
    )
    print("TASK 4.1 - AVG DURATION BY ROUTE TYPE")
    print(avg)
    return avg

### Task 4.2: Top 10 stations by number of distinct trains.

In [25]:
def task_4_2_high_traffic_stations(df):
    top = (
        df.groupby("Station_Name")["Train_No"].nunique()
        .sort_values(ascending=False).head(10)
    )
    print("\nTASK 4.2 - TOP 10 HIGH-TRAFFIC STATIONS")
    print(top)
    return top

### Task 4.3: Save two charts.

In [26]:
def task_4_3_visualizations(avg, top):
    plt.figure(figsize=(6, 4))
    bars = plt.bar(avg.index, avg.values, color=["#4C9AFF", "#F5A623", "#D0021B"])
    plt.title("Average Journey Duration by Route Type")
    plt.ylabel("Hours")
    for bar, val in zip(bars, avg.values):
        plt.text(bar.get_x() + bar.get_width()/2, val + 0.5, f"{val:.1f}h", ha="center")
    plt.tight_layout()
    
    plt.savefig(f"{chart_dir}/avg_duration_by_route_type.png", dpi=100)
    plt.close()

    plt.figure(figsize=(6, 4))
    sns.barplot(x=top.values, y=top.index, color="#2E86AB")
    plt.title("Top 10 High-Traffic Stations")
    plt.xlabel("Number of Trains")
    plt.tight_layout()
    plt.savefig(f"{chart_dir}/top_10_high_traffic_stations.png", dpi=100)
    plt.close()
    print("\nTASK 4.3 - Charts saved to charts/ folder")

### Task 4.4: Print key observations.

In [27]:
def task_4_4_summarize(avg, top):
    print("\nTASK 4.4 - KEY OBSERVATIONS")
    print(f"- Long routes average {avg['Long']}h vs {avg['Short']}h for short routes.")
    print(f"- Busiest station: {top.index[0]} with {top.iloc[0]} trains.")

if __name__ == "__main__":
    df = pd.read_csv(clean_path)
    durations = pd.read_csv(durations_path)
    avg = task_4_1_avg_duration_by_route_type(durations)
    top = task_4_2_high_traffic_stations(df)
    task_4_3_visualizations(avg, top)
    task_4_4_summarize(avg, top)

TASK 4.1 - AVG DURATION BY ROUTE TYPE
Route_Type
Short      1.54
Medium     7.10
Long      27.06
Name: Journey_Duration_Hours, dtype: float64

TASK 4.2 - TOP 10 HIGH-TRAFFIC STATIONS
Station_Name
CST-MUMBAI      1027
KALYAN JN        828
THANE            796
SEALDAH          745
CHENNAI BEAC     738
HOWRAH JN.       699
DADAR            598
DUM DUM JN.      463
KURLA            462
TAMBARAM         434
Name: Train_No, dtype: int64

TASK 4.3 - Charts saved to charts/ folder

TASK 4.4 - KEY OBSERVATIONS
- Long routes average 27.06h vs 1.54h for short routes.
- Busiest station: CST-MUMBAI with 1027 trains.


## Level 5 — Advanced Analysis and Visualization

### Task 5.1: Pivot table - distinct trains & stop records per station.

In [28]:
def task_5_1_pivot_station_distribution(df):
    pivot = pd.pivot_table(
        df, index="Station_Name", values=["Train_No", "SN"],
        aggfunc={"Train_No": pd.Series.nunique, "SN": "count"},
    ).rename(columns={"Train_No": "Distinct_Trains", "SN": "Total_Stop_Records"})
    pivot = pivot.sort_values("Distinct_Trains", ascending=False)
    print("TASK 5.1 - STATION PIVOT TABLE")
    print(pivot.head(10))
    return pivot

### Task 5.2: Cross-tab of top stations vs route type.

In [29]:
def task_5_2_crosstab_station_route_type(merged):
    top_stations = (
        merged.groupby("Station_Name")["Train_No"].nunique()
        .sort_values(ascending=False).head(10).index
    )
    subset = merged[merged["Station_Name"].isin(top_stations)]
    crosstab = pd.crosstab(subset["Station_Name"], subset["Route_Type"])
    crosstab = crosstab.reindex(top_stations)
    cols = [c for c in ["Short", "Medium", "Long"] if c in crosstab.columns]
    crosstab = crosstab[cols]
    print("\nTASK 5.2 - STATION x ROUTE TYPE CROSSTAB")
    print(crosstab)
    return crosstab

### Task 5.3: Comparative charts for the pivot and crosstab.

In [30]:
def task_5_3_visualize(pivot, crosstab):
    top10 = pivot.head(10)
    x = range(len(top10))
    width = 0.4

    plt.figure(figsize=(6, 4))
    plt.bar([i - width/2 for i in x], top10["Distinct_Trains"], width=width, label="Distinct Trains", color="#2E86AB")
    plt.bar([i + width/2 for i in x],top10["Total_Stop_Records"],width=width,label="Total Stop Records",color="#F5A623")
    plt.xticks(list(x), top10.index, rotation=45, ha="right")
    plt.title("Distinct Trains vs Total Stop Records - Top 10 Stations")
    plt.ylabel("Count")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{chart_dir}/pivot_station_distribution.png", dpi=100)
    plt.close()

    plt.figure(figsize=(6, 4))
    crosstab.plot(kind="bar", stacked=True, color=["#4C9AFF", "#F5A623", "#D0021B"], ax=plt.gca())
    plt.title("Route Type Mix at Top 10 Busiest Stations")
    plt.ylabel("Number of Trains")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{chart_dir}/crosstab_station_route_type.png", dpi=100)
    plt.close()

    print("\nTASK 5.3 - Charts saved to charts/ folder")

### Task 5.4: Print advanced insights.

In [31]:
def task_5_4_summarize(pivot, crosstab):
    top_station = pivot.index[0]
    dominant_type = crosstab.loc[top_station].idxmax()
    print("\nTASK 5.4 - ADVANCED INSIGHTS")
    print(f"- {top_station} is the busiest hub; its dominant route type is {dominant_type}.")

if __name__ == "__main__":
    df = pd.read_csv(clean_path)
    durations = pd.read_csv(durations_path)
    merged = df.merge(durations[["Train_No", "Route_Type"]], on="Train_No", how="left")
    pivot = task_5_1_pivot_station_distribution(df)
    crosstab = task_5_2_crosstab_station_route_type(merged)
    task_5_3_visualize(pivot, crosstab)
    task_5_4_summarize(pivot, crosstab)

TASK 5.1 - STATION PIVOT TABLE
              Total_Stop_Records  Distinct_Trains
Station_Name                                     
CST-MUMBAI                  1027             1027
KALYAN JN                    828              828
THANE                        796              796
SEALDAH                      745              745
CHENNAI BEAC                 738              738
HOWRAH JN.                   699              699
DADAR                        598              598
DUM DUM JN.                  463              463
KURLA                        462              462
TAMBARAM                     434              434

TASK 5.2 - STATION x ROUTE TYPE CROSSTAB
Route_Type    Short  Medium  Long
Station_Name                     
CST-MUMBAI      930      29    68
KALYAN JN       577      30   221
THANE           642      43   111
SEALDAH         683      24    38
CHENNAI BEAC    738       0     0
HOWRAH JN.      464      83   152
DADAR           468      44    86
DUM DUM JN.     447  

## Level 6 — Interactive Train Route Enquiry System

In [32]:
class TrainEnquirySystem:
    def __init__(self, data_path=clean_path):
        self.df = pd.read_csv(data_path)
        self.df["Arrival_td"] = pd.to_timedelta(self.df["Arrival_time"])
        self.df["Departure_td"] = pd.to_timedelta(self.df["Departure_Time"])
        self.df["Station_Name_norm"] = (
            self.df["Station_Name"].str.strip().str.upper()
        )
        self._station_lookup = sorted(
            self.df["Station_Name_norm"].unique()
        )

    def station_exists(self, name):
        return name.strip().upper() in self._station_lookup

    def suggest_stations(self, partial, limit=5):
        partial = partial.strip().upper()
        return [
            s for s in self._station_lookup
            if partial in s
        ][:limit]

    def find_direct_trains(self, source, destination):
        source = source.strip().upper()
        destination = destination.strip().upper()

        src = self.df[
            self.df["Station_Name_norm"] == source
        ][
            ["Train_No", "SN", "Departure_td", "Departure_Time"]
        ].rename(columns={
            "SN": "Src_SN",
            "Departure_td": "Src_Departure_td",
            "Departure_Time": "Src_Departure_Time"
        })

        dst = self.df[
            self.df["Station_Name_norm"] == destination
        ][
            ["Train_No", "SN", "Arrival_td", "Arrival_time"]
        ].rename(columns={
            "SN": "Dst_SN",
            "Arrival_td": "Dst_Arrival_td",
            "Arrival_time": "Dst_Arrival_Time"
        })

        merged = src.merge(
            dst,
            on="Train_No",
            how="inner"
        )

        direct = merged[
            merged["Dst_SN"] > merged["Src_SN"]
        ].copy()

        if direct.empty:
            return direct

        def leg_duration(row):
            dep = row["Src_Departure_td"]
            arr = row["Dst_Arrival_td"]

            if pd.isna(dep) or pd.isna(arr):
                return pd.NaT

            if arr < dep:
                arr = arr + pd.Timedelta(days=1)

            return arr - dep

        direct["Leg_Duration"] = direct.apply(
            leg_duration,
            axis=1
        )

        direct["Estimated_Duration"] = direct[
            "Leg_Duration"
        ].apply(_fmt)

        direct = direct.sort_values(
            "Src_Departure_td"
        )

        return direct[
            [
                "Train_No",
                "Src_Departure_Time",
                "Dst_Arrival_Time",
                "Estimated_Duration"
            ]
        ].rename(columns={
            "Src_Departure_Time": "Departure_From_Source",
            "Dst_Arrival_Time": "Arrival_At_Destination"
        }).reset_index(drop=True)
    def find_earliest_trains(self, source, destination, limit=5):
        results = self.find_direct_trains(source, destination)
        return results.head(limit)

In [33]:
def _fmt(td):
    if pd.isna(td):
        return "Unknown"

    total_minutes = int(td.total_seconds() // 60)
    h, m = divmod(total_minutes, 60)

    return f"{h}h {m}m"

In [34]:
def print_station_suggestions(system, station):
    suggestions = system.suggest_stations(station)

    if suggestions:
        print("\nStation not found. Did you mean:")
        for i, suggestion in enumerate(suggestions, start=1):
            print(f"  {i}. {suggestion}")
    else:
        print("\nStation not found. No suggestions available.")

In [35]:
def run_cli(system):
    print("=" * 60)
    print(" TRAIN ROUTE ENQUIRY SYSTEM  (type 'exit' to quit)")
    print("=" * 60)
    while True:
        source = input("Enter SOURCE station: ").strip()
        if source.lower() == "exit":
            break
        if not system.station_exists(source):
            print_station_suggestions(system, source)
            continue

        destination = input("Enter DESTINATION station: ").strip()
        if destination.lower() == "exit":
            break
        if not system.station_exists(destination):
            print_station_suggestions(system, destination)
            continue

        results = system.find_direct_trains(source, destination)
        if source.strip().upper() == destination.strip().upper():
            print("\nSource and destination cannot be the same.\n")
            continue
        if results.empty:
            print(f"\nNo direct trains from {source.upper()} to {destination.upper()}.\n")
        else:
            print(f"\nDirect trains from {source.upper()} to {destination.upper()}:\n")
            print(f"Total direct trains found: {len(results)}\n")
            print("1. Show earliest 5 trains")
            print("2. Show all trains")

            choice = input("\nEnter your choice (1/2): ").strip()

        if choice == "1":
            display_results = results.head(5)

            print("\nEARLIEST 5 TRAINS:")
            print(display_results.to_string(index=False))

        elif choice == "2":
            print("\nALL DIRECT TRAINS:")
            print(results.to_string(index=False))

        else:
            print("\nInvalid choice. Showing earliest 5 trains:")
            print(results.head(5).to_string(index=False))
        print()

        if input("Search again? (y/n): ").strip().lower() != "y":
            break
    print("Goodbye!")

In [36]:
if __name__ == "__main__":
    system = TrainEnquirySystem()
    run_cli(system)

 TRAIN ROUTE ENQUIRY SYSTEM  (type 'exit' to quit)


Enter SOURCE station:  cst-mumbai
Enter DESTINATION station:  kalyan jn



Direct trains from CST-MUMBAI to KALYAN JN:

Total direct trains found: 109

1. Show earliest 5 trains
2. Show all trains



Enter your choice (1/2):  1



EARLIEST 5 TRAINS:
 Train_No Departure_From_Source Arrival_At_Destination Estimated_Duration
    11093              00:10:00               01:12:00              1h 2m
     1011              00:20:00               01:32:00             1h 12m
    51153              05:25:00               06:20:00             0h 55m
    22105              05:40:00               06:33:00             0h 53m
    12859              06:00:00               06:52:00             0h 52m



...


Direct trains from CST-MUMBAI to KALYAN JN:

Total direct trains found: 109

1. Show earliest 5 trains
2. Show all trains



Enter your choice (1/2):  1



EARLIEST 5 TRAINS:
 Train_No Departure_From_Source Arrival_At_Destination Estimated_Duration
    11093              00:10:00               01:12:00              1h 2m
     1011              00:20:00               01:32:00             1h 12m
    51153              05:25:00               06:20:00             0h 55m
    22105              05:40:00               06:33:00             0h 53m
    12859              06:00:00               06:52:00             0h 52m



Search again? (y/n):  n


Goodbye!


In [37]:
system = TrainEnquirySystem()

print("Number of stations:", len(system._station_lookup))
print(system._station_lookup[:20])

Number of stations: 8099
['.BAGHAJATIN', 'ABADA', 'ABHAIPUR', 'ABHANPUR JN.', 'ABHAYAPURI A', 'ABJUGANJ', 'ABOHAR', 'ABU ROAD', 'ABUTARA HALT', 'ACHAL GANJ', 'ACHALDA', 'ACHARYA NARE', 'ACHEGAON', 'ACHHNERA JN.', 'ADALPAHARI', 'ADAPUR', 'ADARKI', 'ADARSH NAGAR', 'ADARSHNAGAR', 'ADAS ROAD']
